In [0]:
%pip install phonenumbers pycountry rapidfuzz

In [0]:
# imports
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StringType
import unicodedata
import re
import phonenumbers
import pycountry
from rapidfuzz import process

In [0]:
#Lecture Bronze
df = spark.table("workspace.final_project.bronze_suppliers")
display(df)

In [0]:
#Profiling rapide
null_counts = df.select([
    F.count(
        F.when(
            F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""),
            c
        )
    ).alias(c)
    for c in df.columns
])

display(null_counts)

#doublons exacts
total_rows = df.count()
distinct_rows = df.distinct().count()

print("Total rows:", total_rows)
print("Distinct rows:", distinct_rows)
print("Exact duplicates:", total_rows - distinct_rows)
print("Exact duplicates:")

#Doublons exacts
duplicate_rows = df.groupBy(df.columns[1:]).count().filter(F.col("count") > 1).drop("count")
display(duplicate_rows)


In [0]:
#Fonction de normalisation
#trim + suppression accents + remplacement caractères spéciaux + inticap

def normalize_string(value):
    if value is None:
        return None

    value = str(value).strip()
    if value == "":
        return None

    # suppression accents
    value = unicodedata.normalize("NFKD", value).encode("ASCII", "ignore").decode("utf-8")

    # remplacement caractères spéciaux par espace
    value = re.sub(r"[^A-Za-z0-9\s\-']", " ", value)

    # espaces multiples
    value = re.sub(r"\s+", " ", value).strip()

    if value == "":
        return None

    return value.title()
